# Bronze Layer: Kafka to S3 Delta Lake Ingestion

This notebook implements the ingestion of raw events from Confluent Kafka into the Bronze Delta table on S3.

### Objectives:
1. Read from Kafka topic `saas.events.raw`.
2. Add metadata columns (`ingested_at`, `kafka_offset`, etc.).
3. Perform basic Data Quality (DQ) checks.
4. Write to S3 Delta Lake in append mode with checkpointing.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

KAFKA_BOOTSTRAP_SERVERS = dbutils.secrets.get(scope="saas-analytics", key="KAFKA_BOOTSTRAP_SERVERS")
KAFKA_API_KEY           = dbutils.secrets.get(scope="saas-analytics", key="KAFKA_API_KEY")
KAFKA_API_SECRET        = dbutils.secrets.get(scope="saas-analytics", key="KAFKA_API_SECRET")
KAFKA_TOPIC             = "saas.events.raw"

aws_access_key = dbutils.secrets.get(scope="saas-analytics", key="AWS_ACCESS_KEY_ID")
aws_secret_key = dbutils.secrets.get(scope="saas-analytics", key="AWS_SECRET_ACCESS_KEY")

spark.conf.set("fs.s3a.access.key",  aws_access_key)
spark.conf.set("fs.s3a.secret.key",  aws_secret_key)
spark.conf.set("fs.s3a.endpoint",    "s3.ap-south-1.amazonaws.com")
spark.conf.set("fs.s3a.impl",        "org.apache.hadoop.fs.s3a.S3AFileSystem")

S3_BUCKET      = "realtime-saas-analytics-pipeline"
BRONZE_PATH    = f"s3://{S3_BUCKET}/bronze/events/"
CHECKPOINT_PATH = f"s3://{S3_BUCKET}/checkpoints/bronze/"

print("✅ Config loaded")

✅ Config loaded


## 1. Read Stream from Kafka
We read the binary data from Kafka. Note: Since we are using Avro, we will deserialize it in the next step.

In [0]:
kafka_options = {
    "kafka.bootstrap.servers": KAFKA_BOOTSTRAP_SERVERS,
    "subscribe":               KAFKA_TOPIC,
    "startingOffsets":         "earliest",
    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism":    "PLAIN",
    "kafka.sasl.jaas.config":  f'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username="{KAFKA_API_KEY}" password="{KAFKA_API_SECRET}";',
}

raw_df = spark.readStream \
    .format("kafka") \
    .options(**kafka_options) \
    .load()

print("✅ Kafka stream connected")

✅ Kafka stream connected


## 2. Add Metadata and Transformation
Adding ingestion timestamp and Kafka metadata.

In [0]:
bronze_df = raw_df.select(
    col("value").cast("binary"),
    current_timestamp().alias("ingested_at"),
    col("offset").alias("kafka_offset"),
    col("partition").alias("kafka_partition"),
    col("topic").alias("kafka_topic"),
    to_date(current_timestamp()).alias("processing_date")
)

## 3. Write Stream to S3 Delta
Partitioned by `processing_date` for efficiency.

In [0]:
query = bronze_df.writeStream \
    .format("delta") \
    .partitionBy("processing_date") \
    .option("checkpointLocation", CHECKPOINT_PATH) \
    .trigger(processingTime="30 seconds") \
    .start(BRONZE_PATH)

print("✅ Bronze stream started — writing to S3 Delta")
print("   Leave this running. Start 02_silver_transform next.")

✅ Bronze stream started — writing to S3 Delta
   Leave this running. Start 02_silver_transform next.


## Monitor Status (run anytime in a new cell)

In [0]:
print("Stream Status:", query.status)
print("Is Active:", query.isActive)

Stream Status: {'message': 'Stopped', 'isDataAvailable': False, 'isTriggerActive': False}
Is Active: False


## Stop stream ONLY when done (run manually)

In [0]:
# for s in spark.streams.active:
#     s.stop()
# print("All streams stopped ✅")

In [0]:
bronze = spark.read.format("delta").load("s3://realtime-saas-analytics-pipeline/bronze/events/")
print("Bronze total:", bronze.count())
bronze.selectExpr("min(processing_date)", "max(processing_date)").show()

Bronze total: 99950
+--------------------+--------------------+
|min(processing_date)|max(processing_date)|
+--------------------+--------------------+
|          2026-05-01|          2026-05-01|
+--------------------+--------------------+

